In [2]:
import json
import pandas as pd
import tweepy

## Load data

In [3]:
# Load JSON data from a file
with open('tweets_final_top.json') as f:
    data1 = json.load(f)

with open('tweets_final.json') as f:
    data2 = json.load(f)

# Convert lists of dictionaries into DataFrames
df1 = pd.DataFrame(data1)
df2 = pd.DataFrame(data2)

# Combine the two DataFrames
combined_df = pd.concat([df1, df2], ignore_index=True)

# Remove duplicates based on the specified columns
df = combined_df.drop_duplicates(subset=['created_at', 'username', 'Text', 'Competitor'])



In [10]:
# Display the DataFrame's columns
df.describe()





,created_at,username,Text,Competitor
count,645,645,645,645
unique,564,431,565,22
top,Wed Aug 07 10:14:10 +0000 2024,FloTrack,[🏃‍♂️] Les athlètes qualifiés en finale du 5 0...,Oscar Chelimo
freq,7,9,7,40


In [11]:
competitor_counts = df.groupby('Competitor').size()

print(competitor_counts)

Competitor
Addisu Yihune               31
Biniam Mehary               19
Dawit Seare                 21
Dominic Lokinyomo Lobalu    18
Edwin Kurgat                27
George Mills                33
Graham Blanks               21
Grant Fisher                38
Hagos Gebrhiwet             18
Hugo Hay                    39
Isaac Kimeli                34
Jacob Krop                  29
Jakob Ingebrigtsen          36
John Heymans                32
Mike Foppen                 33
Narve Gilje Nordas          29
Oscar Chelimo               40
Ronald Kwemoi               27
Stewart Mcsweyn             18
Thierry Ndikumwenayo        35
Thomas Fafard               33
Yann Schrub                 34
dtype: int64


In [13]:
# Load hugging face token
with open('hf_token.json') as file:
    json_obj = json.load(file)
    key=json_obj["Key"]

George Mills and Hugo Hay collided. Mills, Woody Kincaid, Jonah Koech and Birhanu Balew all crahed and were moved into the final because of this disadvantage.

## BERT-Based Models

In [14]:
model = "cardiffnlp/twitter-roberta-base-sentiment-latest"
hf_token = key

Model: cardiffnlp/twitter-roberta-base-sentiment - Specifically fine-tuned on Twitter data for sentiment analysis. It is a RoBERTa-based model that has been trained to classify sentiments into positive, negative, and neutral categories.

Label 0: Negative sentiment
Label 1: Neutral sentiment
Label 2: Positive sentiment

In [16]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import pipeline

# Load the pre-trained model and tokenizer
model_name = "cardiffnlp/twitter-roberta-base-sentiment"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# Create a pipeline for sentiment analysis
sentiment_pipeline = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)

# Function to get sentiment for each tweet
def get_sentiment(text):
    result = sentiment_pipeline(text)
    return result[0]['label'], result[0]['score']

# Apply the sentiment analysis to the "Text" column
df.loc[:, 'Sentiment'], df.loc[:, 'Confidence'] = zip(*df['Text'].apply(get_sentiment))


c:\Users\leaka\anaconda3\lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
c:\Users\leaka\anaconda3\lib\site-packages\pandas\core\indexing.py:1676: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self._setitem_single_column(ilocs[0], value, pi)


In [17]:
# Map sentiment labels to descriptive names
sentiment_labels = {'LABEL_0': 'Negative', 'LABEL_1': 'Neutral', 'LABEL_2': 'Positive'}
df['Sentiment_Label'] = df['Sentiment'].map(sentiment_labels)

# Group by 'Athlete' and 'Sentiment_Label' and count occurrences
sentiment_counts = df.groupby(['Competitor', 'Sentiment_Label']).size().unstack(fill_value=0)

# Rename columns for clarity
sentiment_counts.columns = ['Negative', 'Neutral', 'Positive']



# Calculate total tweets
sentiment_counts['Total_Tweets'] = sentiment_counts.sum(axis=1)

# Calculate ratio of positive tweets
sentiment_counts['Positive_Ratio'] = sentiment_counts['Positive'] / sentiment_counts['Total_Tweets']

sorted_sentiment_counts = sentiment_counts.sort_values(by='Positive_Ratio', ascending=False)

print(sorted_sentiment_counts[['Positive_Ratio']])

                          Negative  Neutral  Positive  Total_Tweets  \
Competitor                                                            
Dawit Seare                      0        9        12            21   
Grant Fisher                     1       19        18            38   
Graham Blanks                    1       12         8            21   
Jakob Ingebrigtsen               5       19        12            36   
Dominic Lokinyomo Lobalu         0       12         6            18   
Ronald Kwemoi                    0       19         8            27   
Edwin Kurgat                     0       20         7            27   
Addisu Yihune                    1       23         7            31   
Oscar Chelimo                    7       24         9            40   
Narve Gilje Nordas               0       23         6            29   
Jacob Krop                       0       23         6            29   
Biniam Mehary                    0       16         3            19   
Thomas

<ipython-input-17-a960e5d58898>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Sentiment_Label'] = df['Sentiment'].map(sentiment_labels)


Model: A BERT-based model fine-tuned for sentiment analysis across multiple languages. It classifies sentiments into five categories: very negative, negative, neutral, positive, and very positive.

In [ ]:
# Load the pre-trained model and tokenizer
model_name = "nlptown/bert-base-multilingual-uncased-sentiment"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# Create a pipeline for sentiment analysis
sentiment_pipeline = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)

# Function to get sentiment for each tweet
def get_sentiment(text):
    result = sentiment_pipeline(text)
    return result[0]['label'], result[0]['score']

# Apply the sentiment analysis to the "Text" column
df.loc[:, 'Sentiment'], df.loc[:, 'Confidence'] = zip(*df['Text'].apply(get_sentiment))

In [23]:

# Group by 'Athlete' and 'Sentiment_Label' and count occurrences
sentiment_counts = df.groupby(['Competitor', 'Sentiment_Label']).size().unstack(fill_value=0)


# Calculate total tweets
sentiment_counts['Total_Tweets'] = sentiment_counts.sum(axis=1)

# Calculate ratio of positive tweets
sentiment_counts['Positive_Ratio'] = sentiment_counts['Positive'] / sentiment_counts['Total_Tweets']

sorted_sentiment_counts = sentiment_counts.sort_values(by='Positive_Ratio', ascending=False)

print(sorted_sentiment_counts[['Positive_Ratio']])

Sentiment_Label           Positive_Ratio
Competitor                              
Dawit Seare                     0.571429
Grant Fisher                    0.473684
Graham Blanks                   0.380952
Jakob Ingebrigtsen              0.333333
Dominic Lokinyomo Lobalu        0.333333
Ronald Kwemoi                   0.296296
Edwin Kurgat                    0.259259
Addisu Yihune                   0.225806
Oscar Chelimo                   0.225000
Narve Gilje Nordas              0.206897
Jacob Krop                      0.206897
Biniam Mehary                   0.157895
Thomas Fafard                   0.151515
John Heymans                    0.125000
Mike Foppen                     0.121212
George Mills                    0.090909
Isaac Kimeli                    0.088235
Hagos Gebrhiwet                 0.055556
Stewart Mcsweyn                 0.055556
Yann Schrub                     0.029412
Thierry Ndikumwenayo            0.028571
Hugo Hay                        0.000000
